# V–RP distance and V–V path composition

This notebook generates the network metrics shown in Fig. 7b and Fig. 7c. It uses the RAGs created from manually annotated ground-truth masks by `05_no_clear_border_202609.ipynb`. Border-touching cells are retained for both analyses.

## Fig. 7b: Vessel-to-nearest-ray distance

Distances are unweighted shortest-path lengths (hops) in the V–AP–RP subgraph. Fibre and, by default, unknown-labelled nodes are excluded.

In [ ]:
# Fig. 7b: vessel-to-nearest-ray distance in the V–AP–RP graph
import os
import re
import glob
import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# Input graphs are produced from manually annotated ground-truth masks by
# 05_no_clear_border_202609.ipynb. Border-touching cells are retained.
GRAPH_DIR = "graphs_vpr_without_clearborder"
OUT_CSV = "plots_dfs/v_to_nearest_ray_distances_with_hops.csv"

# Only vessel, axial parenchyma, and ray-parenchyma nodes are retained.
# Set to True only if unknown-labelled nodes are intended to be traversable.
ALLOW_UNKNOWN = False


def infer_species_from_filename(path):
    """Return the alphabetical species code at the start of a graph filename."""
    name = os.path.basename(path)
    match = re.match(r"([A-Za-z]+)", name)
    return match.group(1).lower() if match else name


def infer_image_id_from_filename(path):
    return re.sub(r"\.gpickle$", "", os.path.basename(path))


os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
records = []
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.gpickle")))

if not graph_files:
    raise FileNotFoundError(f"No graph files were found in {GRAPH_DIR}")

for graph_path in graph_files:
    with open(graph_path, "rb") as handle:
        G = pickle.load(handle)

    allowed = {"vessel", "ray", "parenchyma"}
    if ALLOW_UNKNOWN:
        allowed.add("unknown")

    keep_nodes = [
        node for node, data in G.nodes(data=True)
        if data.get("anatomy") in allowed
    ]
    H = G.subgraph(keep_nodes).copy()

    vessels = [
        node for node, data in H.nodes(data=True)
        if data.get("anatomy") == "vessel"
    ]
    rays = [
        node for node, data in H.nodes(data=True)
        if data.get("anatomy") == "ray"
    ]

    if not vessels or not rays:
        print(
            f"Skipping {os.path.basename(graph_path)}: "
            f"vessels={len(vessels)}, rays={len(rays)}"
        )
        continue

    # Every edge has unit cost: the reported distance is an unweighted hop count.
    try:
        dist_hops = nx.multi_source_dijkstra_path_length(
            H, sources=rays, weight=lambda u, v, data: 1
        )
    except Exception:
        # Fallback for graph types for which the weighted interface is unavailable.
        Hu = nx.Graph()
        Hu.add_nodes_from(H.nodes(data=True))
        Hu.add_edges_from(H.edges())
        dist_hops = nx.multi_source_shortest_path_length(Hu, sources=rays)

    species = infer_species_from_filename(graph_path)
    image_id = infer_image_id_from_filename(graph_path)

    for vessel in vessels:
        records.append(
            {
                "species": species,
                "image_id": image_id,
                "node_id": vessel,
                # NaN indicates that no ray is reachable in the V–AP–RP subgraph.
                "dist_hops": dist_hops.get(vessel, np.nan),
            }
        )

df_hops = pd.DataFrame(records)
if df_hops.empty:
    raise ValueError("No vessel-to-ray distances were calculated.")

df_hops.to_csv(OUT_CSV, index=False)
print(f"Saved {len(df_hops)} vessel records to {OUT_CSV}")
display(df_hops.groupby("species")["dist_hops"].describe())

In [ ]:
# Plot the Fig. 7b distribution of unweighted V–RP distances.
df_h = pd.read_csv("plots_dfs/v_to_nearest_ray_distances_with_hops.csv")
df_h["is_unreachable"] = ~np.isfinite(df_h["dist_hops"])

species_order = sorted(df_h["species"].dropna().unique())
finite = df_h.loc[np.isfinite(df_h["dist_hops"]), "dist_hops"]
if finite.empty:
    raise ValueError("No finite hop distances are available for plotting.")

hop_min = int(finite.min())
hop_max = int(finite.max())
bins = np.arange(hop_min - 0.5, hop_max + 1.5, 1)
x_unreachable = hop_max + 2.5

fig, axes = plt.subplots(
    len(species_order), 1, figsize=(6, 1.8 * len(species_order)), sharex=True
)
if len(species_order) == 1:
    axes = [axes]

for ax, species in zip(axes, species_order):
    subset = df_h[df_h["species"] == species]
    values = subset.loc[np.isfinite(subset["dist_hops"]), "dist_hops"].astype(int).values

    if values.size:
        counts, edges = np.histogram(values, bins=bins)
        ax.bar(
            edges[:-1], counts, width=np.diff(edges), align="edge",
            edgecolor="white", linewidth=0.5, color="tab:blue",
        )
        ax.axvline(np.mean(values), color="darkgrey", linestyle="--", linewidth=1)
        bar_width = float(np.diff(edges)[0])
    else:
        bar_width = 1.0

    n_unreachable = int(subset["is_unreachable"].sum())
    ax.bar(
        x_unreachable, n_unreachable, width=bar_width, align="center",
        color="tab:grey", alpha=0.6, edgecolor="white",
    )
    ax.text(
        x_unreachable, n_unreachable, str(n_unreachable),
        ha="center", va="bottom", fontsize=9, color="tab:grey",
    )
    ax.set_ylabel(species, rotation=0, labelpad=30, va="center")

all_ticks = list(range(hop_min, hop_max + 1)) + [x_unreachable]
all_labels = [str(value) for value in range(hop_min, hop_max + 1)] + ["Unreachable"]
for ax in axes:
    ax.set_xticks(all_ticks)
    ax.set_xticklabels(all_labels)

axes[-1].set_xlim(bins[0], x_unreachable + 1.2)
axes[-1].set_xlabel("Graph distance (hops)")
mpl.rcParams["pdf.fonttype"] = 42
plt.tight_layout(h_pad=0.2)
plt.savefig("plots_dfs/v_to_nearest_ray_hops.pdf")
plt.savefig("plots_dfs/v_to_nearest_ray_hops.png", dpi=300)
plt.show()

## Fig. 7c: Vessel-to-vessel path composition

For each connected component, up to 20 source vessels and up to 25 target vessels per source are sampled with random seed 42. The internal nodes of unweighted shortest paths are used to calculate composition.

In [ ]:
# Fig. 7c: cellular composition of sampled vessel-to-vessel shortest paths
import os
import re
import glob
import pickle
import random

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

GRAPH_DIR = "graphs_vpr_without_clearborder"
OUT_CSV = "plots_dfs/vv_shortest_paths_composition_LITE.csv"
OUT_FIGURE = "plots_dfs/vv_avg_composition_by_species_LITE.pdf"

RANDOM_SEED = 42
MAX_SOURCES_PER_GRAPH = 20
TARGETS_PER_SOURCE = 25
INCLUDE_ENDS = False
ALLOW_UNKNOWN = False

CATEGORIES = ["vessel", "ray", "parenchyma", "unknown"]
PLOT_ORDER = ["ray", "parenchyma", "vessel"]
CATEGORY_COLORS = {
    "parenchyma": "lightcoral",
    "ray": "royalblue",
    "vessel": "lightgray",
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def species_from_filename(path):
    match = re.match(r"([A-Za-z]+)", os.path.basename(path))
    return match.group(1).lower() if match else os.path.basename(path)


def image_id_from_filename(path):
    return os.path.basename(path).replace(".gpickle", "")


def simple_undirected(graph):
    undirected = nx.Graph()
    undirected.add_nodes_from(graph.nodes(data=True))
    undirected.add_edges_from(graph.edges())
    return undirected


def rebuild_path_from_predecessor(predecessors, source, target):
    path = [target]
    current = target
    while current != source:
        previous = predecessors.get(current, [])
        if not previous:
            return None
        current = previous[0]
        path.append(current)
    return list(reversed(path))


def path_categories(graph, path):
    internal_nodes = path if INCLUDE_ENDS else path[1:-1]
    return [
        graph.nodes[node].get("anatomy")
        if graph.nodes[node].get("anatomy") in CATEGORIES
        else "unknown"
        for node in internal_nodes
    ]


def composition_fractions(counts):
    total = sum(counts.get(category, 0) for category in CATEGORIES)
    if total == 0:
        return {category: np.nan for category in CATEGORIES}
    return {category: counts.get(category, 0) / total for category in CATEGORIES}


os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
records = []
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.gpickle")))
if not graph_files:
    raise FileNotFoundError(f"No graph files were found in {GRAPH_DIR}")

for graph_path in graph_files:
    with open(graph_path, "rb") as handle:
        G_raw = pickle.load(handle)

    G = simple_undirected(G_raw)
    allowed = {"vessel", "ray", "parenchyma"}
    if ALLOW_UNKNOWN:
        allowed.add("unknown")
    keep_nodes = [
        node for node, data in G.nodes(data=True)
        if data.get("anatomy") in allowed
    ]
    H = G.subgraph(keep_nodes).copy()

    vessels = [
        node for node, data in H.nodes(data=True)
        if data.get("anatomy") == "vessel"
    ]
    species = species_from_filename(graph_path)
    image_id = image_id_from_filename(graph_path)

    for component in nx.connected_components(H):
        component_vessels = [vessel for vessel in vessels if vessel in component]
        if len(component_vessels) < 2:
            continue

        sources = random.sample(
            component_vessels,
            min(len(component_vessels), MAX_SOURCES_PER_GRAPH),
        )
        for source in sources:
            predecessors, _ = nx.predecessor(H, source=source, return_seen=True)
            reachable_targets = [
                vessel for vessel in component_vessels
                if vessel != source and vessel in predecessors
            ]
            if not reachable_targets:
                continue

            targets = random.sample(
                reachable_targets,
                min(len(reachable_targets), TARGETS_PER_SOURCE),
            )
            for target in targets:
                path = rebuild_path_from_predecessor(predecessors, source, target)
                if not path:
                    continue
                categories = path_categories(H, path)
                if not categories:
                    continue

                counts = {category: categories.count(category) for category in CATEGORIES}
                fractions = composition_fractions(counts)
                record = {
                    "species": species,
                    "image_id": image_id,
                    "src": source,
                    "dst": target,
                    "path_len": max(len(path) - 1, 0),
                }
                record.update({f"count_{key}": value for key, value in counts.items()})
                record.update({f"frac_{key}": value for key, value in fractions.items()})
                records.append(record)

df_paths = pd.DataFrame(records)
if df_paths.empty:
    raise ValueError("No vessel-to-vessel paths were sampled.")

df_paths.to_csv(OUT_CSV, index=False)
print(f"Saved {len(df_paths)} sampled paths to {OUT_CSV}")

df_mean = (
    df_paths.groupby("species")[[f"frac_{category}" for category in CATEGORIES]]
    .mean()
    .rename(columns={f"frac_{category}": category for category in CATEGORIES})
)

fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(df_mean)), 6))
bottom = np.zeros(len(df_mean))
x = np.arange(len(df_mean))
for category in PLOT_ORDER:
    values = df_mean[category].fillna(0).values
    bars = ax.bar(
        x, values, bottom=bottom, label=category,
        color=CATEGORY_COLORS.get(category, "gray"),
    )
    labels = [f"{value * 100:.1f}%" if value >= 0.03 else "" for value in values]
    ax.bar_label(bars, labels=labels, label_type="center", fontsize=8, color="black")
    bottom += values

mpl.rcParams["pdf.fonttype"] = 42
ax.set_xticks(x, df_mean.index)
ax.set_ylabel("Average fraction along V–V shortest paths")
ax.legend(title="Cell type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUT_FIGURE)
plt.savefig(OUT_FIGURE.with_suffix(".png"), dpi=300)
plt.show()

## Table Sx: Structure and graph counts

This cell summarises vessels, AP cells, RP cells, V–AP–RP RAG nodes and edges, and sampled V–V paths for each performance-evaluation region.

In [ ]:
# Table Sx: numbers of annotated structures, graph elements, and sampled paths
# Counts refer to the border-retaining V–AP–RP RAG used in Fig. 7b and Fig. 7c.
import glob
import os
import pickle

import networkx as nx
import pandas as pd

GRAPH_DIR = "graphs_vpr_without_clearborder"
PATH_TABLE = "plots_dfs/vv_shortest_paths_composition_LITE.csv"
OUT_TABLE = "plots_dfs/performance_evaluation_region_structure_counts.csv"


def structure_species_from_filename(path):
    match = re.match(r"([A-Za-z]+)", os.path.basename(path))
    return match.group(1).lower() if match else os.path.basename(path)


def structure_image_id_from_filename(path):
    return os.path.basename(path).replace(".gpickle", "")


if not os.path.exists(PATH_TABLE):
    raise FileNotFoundError(
        f"{PATH_TABLE} was not found. Run the Fig. 7c cell before this cell."
    )

path_counts = (
    pd.read_csv(PATH_TABLE)
    .groupby("image_id")
    .size()
    .rename("n_sampled_vv_paths")
)

records = []
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.gpickle")))
if not graph_files:
    raise FileNotFoundError(f"No graph files were found in {GRAPH_DIR}")

for graph_path in graph_files:
    with open(graph_path, "rb") as handle:
        G_raw = pickle.load(handle)

    allowed = {"vessel", "ray", "parenchyma"}
    keep_nodes = [
        node for node, data in G_raw.nodes(data=True)
        if data.get("anatomy") in allowed
    ]
    H = G_raw.subgraph(keep_nodes).copy()

    image_id = structure_image_id_from_filename(graph_path)
    anatomy = nx.get_node_attributes(H, "anatomy")
    records.append(
        {
            "species": structure_species_from_filename(graph_path),
            "image_id": image_id,
            "n_vessels": sum(value == "vessel" for value in anatomy.values()),
            "n_ap_cells": sum(value == "parenchyma" for value in anatomy.values()),
            "n_rp_cells": sum(value == "ray" for value in anatomy.values()),
            "n_rag_nodes": H.number_of_nodes(),
            "n_rag_edges": H.number_of_edges(),
            "n_sampled_vv_paths": int(path_counts.get(image_id, 0)),
        }
    )

structure_counts = pd.DataFrame(records).sort_values("species").reset_index(drop=True)
os.makedirs(os.path.dirname(OUT_TABLE), exist_ok=True)
structure_counts.to_csv(OUT_TABLE, index=False)

print(f"Saved Table Sx source data to {OUT_TABLE}")
display(structure_counts)